# 04 · Results

Evaluate the strategy **exactly as configured in `config.yaml`**. This notebook decides nothing: if a result here
makes you want to change a parameter, that change belongs in 02/03 on train data, and the test result shown
here is then no longer a clean out-of-sample result. Say so in the report if it happens.

This is the **only** notebook that uses the test period (`period="test"`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Make the repo root importable, whether Jupyter starts in notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data, strategy, backtest, plots

plots.apply_style()
pd.set_option("display.float_format", "{:.4f}".format)
cfg = data.load_config()

In [ ]:
prices = data.load_prices(cfg, verbose=False)
funding = data.load_funding(cfg, coins=list(prices.columns))
pair = tuple(cfg["parameters"]["pair"])
params = data.get_params(cfg)
label = f"{pair[0]}/{pair[1]}"

print(f"Pair: {label}")
print({k: v for k, v in params.items() if k != "train_end"})

## 1. In-sample results (train)

In [ ]:
train_res = backtest.run_strategy(prices, funding, pair, params, period="train")
display(plots.plot_equity_curve(train_res, title=f"{label}: equity, train (in-sample)", save="04_equity_train.png"))
display(plots.plot_drawdown(train_res, title=f"{label}: drawdown, train", save="04_drawdown_train.png"))
display(plots.plot_spread_zscore(train_res, save="04_spread_train.png"))
display(plots.plot_pnl_breakdown(train_res, title=f"{label}: PnL breakdown, train", save="04_pnl_train.png"))
display(plots.plot_trade_distribution(train_res))
pd.Series(train_res.metrics).to_frame("train")

## 2. Trade walkthrough

Three trades from the train period: the best, the worst, and one closed by a stop (if any). For each, explain in
the report why it was opened (z beyond the entry threshold), what the two legs did, and why it was closed.

In [ ]:
trades = train_res.trades
walkthrough = {"best trade": trades["pnl_pct"].idxmax(), "worst trade": trades["pnl_pct"].idxmin()}
stopped = trades.index[trades["exit_reason"].isin(["z_stop", "time_stop", "coint_break"])]
if len(stopped):
    walkthrough["stopped trade"] = stopped[0]

for name, i in walkthrough.items():
    display(plots.plot_trade(train_res, int(i), save=f"04_{name.replace(' ', '_')}.png"))
trades.loc[list(walkthrough.values())].assign(case=list(walkthrough.keys())).set_index("case")

*Interpretation (to fill in):* best trade: … · worst trade: … · stopped trade: …

## 3. Out-of-sample results (test)

The single, final run on data the strategy has never seen. Hedge ratio (if static) and half-life come from train;
rolling quantities warm up on the end of the train period (they only look backwards).

In [ ]:
test_res = backtest.run_strategy(prices, funding, pair, params, period="test")
display(plots.plot_equity_curve(test_res, title=f"{label}: equity, test (out-of-sample)", save="04_equity_test.png"))
display(plots.plot_drawdown(test_res, title=f"{label}: drawdown, test", save="04_drawdown_test.png"))
display(plots.plot_spread_zscore(test_res, save="04_spread_test.png"))
display(plots.plot_pnl_breakdown(test_res, title=f"{label}: PnL breakdown, test", save="04_pnl_test.png"))
plots.plot_trade_distribution(test_res)

## 4. Robustness

Same strategy, harsher or simplified assumptions: fees doubled, slippage doubled, and without funding
(to see how much of the result is funding). A strategy that only works with exactly our cost assumptions is fragile.

In [ ]:
robustness_variants = {
    "base": {},
    "fees x2": {"fee_per_leg": 2 * params["fee_per_leg"]},
    "slippage x2": {"slippage_per_leg": 2 * params["slippage_per_leg"]},
    "without funding": {"use_funding": False},
}
robustness = {}
for period in ["train", "test"]:
    for name, overrides in robustness_variants.items():
        res = backtest.run_strategy(prices, funding, pair, data.get_params(cfg, **overrides), period=period)
        robustness[(period, name)] = res.metrics
robustness_table = pd.DataFrame(robustness).loc[["sharpe", "total_return", "max_drawdown", "total_fees",
                                                 "total_slippage", "total_funding"]]
robustness_table

## 5. In-sample vs out-of-sample

In [ ]:
comparison = backtest.compare_metrics({"train (in-sample)": train_res, "test (out-of-sample)": test_res})
display(plots.plot_equity_curve({"train": train_res.equity / train_res.equity.iloc[0],
                                 "test": test_res.equity / test_res.equity.iloc[0]},
                                title=f"{label}: growth of 1 USDT, train vs test"))
comparison

## 6. Interpretation (to fill in)

- Does the out-of-sample Sharpe confirm the in-sample result, or does it collapse? How large is the drop, and is
  it within what we would expect from noise with this number of trades?
- What drives the PnL (gross vs costs vs funding)?
- Which risk rules mattered in the test period?
- How sensitive is the result to costs (robustness table)?

## 7. Lessons learned (to fill in)

- …
- …
- What we would do next with more time: …